In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import os

In [2]:
df=pd.read_csv(r"C:\Users\heman\Natural_Language_processing\IMDB Dataset.csv")

In [3]:
df=df.iloc[:10000]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df["review"][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [5]:
df["sentiment"].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [6]:
def remove_html(text):
    pattern=re.compile("<.*?>")
    return pattern.sub(r"",text)
df["review"]=df["review"].apply(remove_html)

In [7]:
df["review"]=df["review"].apply(lambda x :x.lower())

In [8]:
import string, time
exclude=string.punctuation

def remove_punctuation(text):
    new_text=[]
    for t in text:
        if t not in exclude:
            new_text.append(t)
    return "".join(new_text)
# remove_punctuation("hemant!# is a good man!")
df["review"]=df["review"].apply(remove_punctuation)

In [9]:
from nltk.corpus import stopwords
# stopwords.words("english")
def remove_stopword(text):
    new_text=[]
    for word in text.split():
        if word not in stopwords.words("english"):
            new_text.append(word)
    return " ".join(new_text)
df["review"]=df["review"].apply(remove_stopword)

In [10]:
df.duplicated().sum()

np.int64(17)

In [11]:
df.drop_duplicates(inplace=True)

In [25]:
x=df["review"]
y=df["sentiment"]

In [26]:
from sklearn.preprocessing import LabelEncoder
LB=LabelEncoder()
y_LB=LB.fit_transform(y)

In [27]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y_LB, test_size=0.2, random_state=42)

# BOW
But CountVectorizer() expects a sequence of strings, not a DataFrame.

In [50]:
from sklearn.feature_extraction.text import CountVectorizer
CV=CountVectorizer(max_features=500)
x_train_BOW=CV.fit_transform(x_train)
x_test_BOW=CV.transform(x_test)

In [51]:
x_train_b=x_train_BOW.toarray()
x_test_b=x_test_BOW.toarray()

In [52]:
x_train_b

array([[0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 1],
       [0, 0, 1, ..., 0, 0, 0]], shape=(7986, 500))

In [53]:
from sklearn.naive_bayes import MultinomialNB
MNB=MultinomialNB()
MNB.fit(x_train_BOW,y_train)

MultinomialNB()

In [54]:
predict=MNB.predict(x_test_BOW)
predict

array([0, 0, 1, ..., 1, 0, 0], shape=(1997,))

In [55]:
from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(predict,y_test)

0.8312468703054582

In [59]:
from sklearn.naive_bayes import GaussianNB
GNB = GaussianNB()
GNB.fit(x_train_b, y_train)
y_pred = GNB.predict(x_test_b)
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7926890335503255


In [61]:
# cv=CountVectorizer(max_feature=500)
from sklearn.ensemble import RandomForestClassifier
rm=RandomForestClassifier()
rm.fit(x_train_b,y_train)
y_pred=rm.predict(x_test_b)
print("Accuracy",accuracy_score(y_pred,y_test))

Accuracy 0.799699549323986


In [64]:
cv=CountVectorizer(ngram_range=(1,2),max_features=1000)
x_train_g=cv.fit_transform(x_train).toarray()
x_test_g=cv.transform(x_test).toarray()
from sklearn.ensemble import RandomForestClassifier
rm=RandomForestClassifier()
rm.fit(x_train_g,y_train)
y_pred=rm.predict(x_test_g)
print("Accuracy",accuracy_score(y_pred,y_test))

Accuracy 0.8187280921382073


# TFIDF
But CountVectorizer() expects a sequence of strings, not a DataFrame.

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
x_train_t=tfidf.fit_transform(x_train)
x_test_t=tfidf.transform(x_test)

from sklearn.ensemble import RandomForestClassifier
rm=RandomForestClassifier()
rm.fit(x_train_t,y_train)
y_pred=rm.predict(x_test_t)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.8522784176264396
